# CoordinatorAgent - Wood Lantern Making

This example demonstrates how to use the `CoordinatorAgent` with the `transformers` subpackage to plan and execute the process of making a wood lantern. The coordinator decomposes the task into five specialized phases handled by different worker agents:

1. **Design Worker** - Conceptual design (style, dimensions, structural plan)
2. **Procurement Worker** - Material sourcing (wood types, tools, adhesives, finishes)
3. **Skeleton Worker** - Frame construction (cutting, joining, assembling the lantern skeleton)
4. **Decoration Worker** - Artistic finishing (carving, painting, paper mounting, lighting)
5. **Quality Assurance Worker** - Safety and durability testing (structural integrity, heat resistance, stability)

The coordinator plans the steps, delegates to workers sequentially, and synthesizes a final comprehensive guide.

In [1]:
from os import environ

environ.setdefault("HF_HOME", "/data/hf_models/")


'/data/hf_models/'

In [2]:
!export HF_TOKEN=$(cat /data/hf_token.txt)

In [ ]:
import json
import re
from typing import List, Sequence, Tuple

from a2a.types import AgentCapabilities, AgentCard, AgentSkill
from aap_core.agent import BaseAgent
from aap_core.orchestration import CoordinatorAgent
from aap_core.types import AgentMessage, BaseLLMChain
from aap_transformers.chain import ChatCausalMultiTurnsChain

from transformers import AutoModelForCausalLM, AutoTokenizer

# Load shared models to save memory
print("Loading shared models...")
shared_model_1 = AutoModelForCausalLM.from_pretrained(
    "swiss-ai/Apertus-8B-Instruct-2509",
    device_map="cuda",
    torch_dtype="auto",
)
shared_model_1.eval()
shared_tokenizer_1 = AutoTokenizer.from_pretrained("swiss-ai/Apertus-8B-Instruct-2509")

shared_model_2 = AutoModelForCausalLM.from_pretrained(
    "HuggingFaceTB/SmolLM3-3B",
    device_map="cuda",
    torch_dtype="auto",
)
shared_model_2.eval()
shared_tokenizer_2 = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM3-3B")

print("Shared models loaded successfully!")


/data/agent_design_pattern/src/transformers/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading shared models...


## Define the Worker Agent

Create a simple worker agent that wraps a chain for execution.


In [ ]:
class WorkerAgent(BaseAgent):
    chain: BaseLLMChain

    def execute(self, message: AgentMessage, **kwargs) -> AgentMessage:
        self.state = "running"
        message = self.chain.invoke(message, **kwargs)
        message.execution_result = "success"
        message.origin = self.card.name
        self.state = "idle"
        return message


## Setup Ollama Connection

We'll use `mistral:7b-instruct-v0.3-q4_K_S` (7.2B parameters) as our LLM backend.


In [ ]:
# State callback for monitoring execution
def state_callback(state: str):
    print(f"[Coordinator] {state}")


## Define the Design Worker Chain

The design worker focuses on conceptual design: style, dimensions, structural plan, and overall lantern concept.


In [ ]:
design_system_prompt = """You are an expert woodworker and lantern designer.
Your task is to create a comprehensive design plan for a wood lantern.

Provide a detailed design that includes:
1. **Style**: Describe the lantern style (e.g., traditional Asian, rustic cabin, modern geometric, fairy garden)
2. **Dimensions**: Specify height, width, depth, and overall proportions
3. **Structure**: Describe the basic shape (cube, hexagonal, pagoda, cylinder, etc.)
4. **Light Source**: Recommend the type of light (LED candle, fairy lights, real candle with safety considerations)
5. **Material Palette**: Suggest wood types and complementary materials (paper, fabric, metal accents)
6. **Design Details**: Include decorative elements, cutout patterns, joinery approach

Format your response with clear sections. Be specific and creative. Include measurements and design specifications.
"""

design_user_prompt = """{query}

Previous context: {context_results}
"""

design_chain = ChatCausalMultiTurnsChain(
    model=(shared_tokenizer_1, shared_model_1),
    system_prompt=design_system_prompt,
    user_prompt_template=design_user_prompt,
    device="cuda",
    max_new_tokens=4096,
)
design_chain.final_response_as_context("design")

## Define the Procurement Worker Chain

The procurement worker focuses on material sourcing: wood types, tools, adhesives, finishes, and hardware needed for the lantern project.


In [ ]:
procurement_system_prompt = """You are an expert woodworker and materials specialist.
Your task is to create a comprehensive procurement list for building a wood lantern based on the design plan.

Provide a detailed materials and tools list that includes:
1. **Wood Materials**: Specific wood types (e.g., basswood, birch plywood, cedar), quantities, and grades
2. **Adhesives**: Wood glue types, clamps needed, drying times
3. **Fasteners**: Screws, nails, dowels, or joinery hardware
4. **Finishes**: Stains, sealants, varnishes, or natural oils (food-safe if applicable)
5. **Cutout Materials**: Translucent paper (rice paper, parchment), fabric, or other light-diffusing materials
6. **Lighting Components**: LED candles, fairy light strings, batteries, wiring
7. **Tools Required**: Saw types, sandpaper grits, drill bits, clamps, measuring tools
8. **Estimated Costs**: Rough cost ranges for each category

Format your response with clear sections. Be specific about quantities and specifications.
"""

procurement_user_prompt = """{query}

Design plan context: {context_design}
"""

procurement_chain = ChatCausalMultiTurnsChain(
    model=(shared_tokenizer_1, shared_model_1),
    system_prompt=procurement_system_prompt,
    user_prompt_template=procurement_user_prompt,
    device="cuda",
    max_new_tokens=4096,
)
procurement_chain.final_response_as_context("procurement")

## Define the Skeleton Worker Chain

The skeleton worker focuses on frame construction: cutting, joining, and assembling the lantern's structural frame.


In [ ]:
skeleton_system_prompt = """You are an expert woodworker specializing in lantern frame construction.
Your task is to provide detailed instructions for building the structural frame of a wood lantern.

Provide a comprehensive construction guide that includes:
1. **Frame Design**: Describe the structural layout (rings, ribs, pillars) based on the lantern shape
2. **Wood Selection**: Recommend specific wood types for frame members (e.g., basswood for thin ribs, birch plywood for bases, bamboo splints for traditional style)
3. **Cutting Instructions**: Specify dimensions for each frame component with tolerances
4. **Joinery Method**: Detail how to connect frame members (dovetail, mortise-and-tenon, dowel, glue-and-clamp, or traditional lashings)
5. **Assembly Sequence**: Step-by-step order for building the frame (base ring → ribs → top ring → vertical pillars)
6. **Traditional Techniques**: Include traditional methods if applicable (e.g., bamboo splint weaving for Teochew style, parallel splints for Fuzhou umbrella style)
7. **Tools & Jigs**: Recommend specific jigs or fixtures for accurate assembly
8. **Common Pitfalls**: Warn about warping, uneven tension, or weak joints

Format your response with clear sections. Include measurements, wood grain orientation notes, and assembly tips.
"""

skeleton_user_prompt = """{query}

Design plan context: {context_design}
Materials list context: {context_procurement}
"""

skeleton_chain = ChatCausalMultiTurnsChain(
    model=(shared_tokenizer_1, shared_model_1),
    system_prompt=skeleton_system_prompt,
    user_prompt_template=skeleton_user_prompt,
    device="cuda",
    max_new_tokens=4096,
)
skeleton_chain.final_response_as_context("skeleton")

## Define the Decoration Worker Chain

The decoration worker focuses on artistic finishing: painting, carving, paper mounting, and adding traditional Chinese decorative elements.


In [ ]:
decoration_system_prompt = """You are an expert artisan specializing in traditional Chinese lantern decoration.
Your task is to provide detailed instructions for decorating and finishing a wood lantern.

Provide a comprehensive decoration guide that includes:
1. **Surface Preparation**: Sanding, priming, and preparing the wood surface for decoration
2. **Paper/Silk Mounting**: Instructions for attaching rice paper, silk, or other translucent materials to the frame (use rice glue for traditional approach, smooth wrinkles carefully)
3. **Traditional Painted Motifs**: Describe and provide painting instructions for classic Chinese lantern patterns:
   - Dragons (power and strength)
   - Phoenixes (grace and prosperity)
   - Peonies (wealth and honor)
   - Tigers (courage and protection)
   - Eight Immortals (traditional legend figures)
   - Family names or clan titles (traditional Fuzhou style)
4. **Gold Details**: Instructions for adding gold calligraphy, gilded patterns, or gold leaf accents
5. **Color Scheme**: Recommend traditional color palettes (red with gold is classic; white with red/gold wordings for temples; black with gold for special occasions)
6. **Tassels & Fringes**: How to attach decorative tassels (red-and-gold) to the base, symbolizing renewal
7. **Hanging Mechanism**: Create a handle or hanging cord (traditional red string tied to a wooden handle)
8. **Light Integration**: How to safely install LED candles or fairy lights inside

Format your response with clear sections. Include step-by-step painting techniques and material specifications.
"""

decoration_user_prompt = """{query}

Design plan context: {context_design}
Materials list context: {context_procurement}
Frame construction context: {context_skeleton}
"""

decoration_chain = ChatCausalMultiTurnsChain(
    model=(shared_tokenizer_2, shared_model_2),
    system_prompt=decoration_system_prompt,
    user_prompt_template=decoration_user_prompt,
    device="cuda",
    max_new_tokens=4096,
)
decoration_chain.final_response_as_context("decoration")

## Define the Quality Assurance Worker Chain

The quality assurance worker focuses on safety and durability testing: structural integrity, heat resistance, stability, and overall finish quality.


In [ ]:
qa_system_prompt = """You are an expert quality assurance inspector specializing in traditional Chinese lanterns.
Your task is to provide a comprehensive quality assurance and testing protocol for a finished wood lantern.

Provide a detailed QA checklist that includes:
1. **Structural Integrity Tests**:
   - Weight-bearing test (can it hang without frame distortion?)
   - Joint strength test (do all connections hold under gentle stress?)
   - Frame squareness/roundness check (is the lantern symmetrical?)
   - Balance test (does it hang level without tilting?)
2. **Heat Resistance Tests** (if using real candles):
   - Distance from flame to paper/silk surface (minimum 3 inches recommended)
   - Heat shield evaluation (is there adequate clearance?)
   - Fire retardant treatment assessment
   - **Strongly recommend LED alternatives** for safety
3. **Material Durability**:
   - Paper/silk adhesion check (no peeling edges?)
   - Wood finish durability (no cracking or flaking?)
   - Tassel attachment security
   - Hanging cord strength
4. **Light Distribution Quality**:
   - Even light diffusion through paper/silk panels
   - No dark spots or overly bright areas
   - Shadow pattern aesthetics when lit
5. **Aesthetic Quality**:
   - Paint quality (no drips, even coverage, crisp edges)
   - Gold detail application (proper adhesion, no tarnishing)
   - Tassel symmetry and alignment
   - Overall visual balance
6. **Safety Recommendations**:
   - Indoor vs outdoor usage guidelines
   - Supervision requirements
   - Distance from flammable materials
   - Weather exposure limitations (traditional lanterns are NOT weatherproof)
7. **Common Defects & Fixes**:
   - Wrinkled paper → light misting and ironing
   - Tilting frame → adjust wire tension
   - Weak joints → reinforce with additional glue/dowels

Format your response with clear sections. Use a checklist format where possible.
"""

qa_user_prompt = """{query}

Design plan context: {context_design}
Materials list context: {context_procurement}
Frame construction context: {context_skeleton}
Decoration guide context: {context_decoration}
"""

qa_chain = ChatCausalMultiTurnsChain(
    model=(shared_tokenizer_2, shared_model_2),
    system_prompt=qa_system_prompt,
    user_prompt_template=qa_user_prompt,
    device="cuda",
    max_new_tokens=4096,
)
qa_chain.final_response_as_context("qa")

## Define Worker Agents

Create the five worker agents using the chains defined above. Each agent wraps a chain and executes it.


In [ ]:
# Design Worker
design_skill = AgentSkill(id='design-skill', name="design skill", description="Conceptual design for wood lanterns", tags=['design', 'lantern'])
design_card = AgentCard(name='design_agent', description='Lantern design specialist', skills=[design_skill], capabilities=AgentCapabilities(), default_input_modes=['text'], default_output_modes=['text'], url='localhost', version='0.1.0')
design_agent = WorkerAgent(chain=design_chain, card=design_card, state_change_callback=state_callback)

# Procurement Worker
procurement_skill = AgentSkill(id='procurement-skill', name="procurement skill", description="Material sourcing for lantern projects", tags=['procurement', 'materials'])
procurement_card = AgentCard(name='procurement_agent', description='Materials sourcing specialist', skills=[procurement_skill], capabilities=AgentCapabilities(), default_input_modes=['text'], default_output_modes=['text'], url='localhost', version='0.1.0')
procurement_agent = WorkerAgent(chain=procurement_chain, card=procurement_card, state_change_callback=state_callback)

# Skeleton Worker
skeleton_skill = AgentSkill(id='skeleton-skill', name="skeleton skill", description="Frame construction for lanterns", tags=['skeleton', 'construction'])
skeleton_card = AgentCard(name='skeleton_agent', description='Frame construction specialist', skills=[skeleton_skill], capabilities=AgentCapabilities(), default_input_modes=['text'], default_output_modes=['text'], url='localhost', version='0.1.0')
skeleton_agent = WorkerAgent(chain=skeleton_chain, card=skeleton_card, state_change_callback=state_callback)

# Decoration Worker
decoration_skill = AgentSkill(id='decoration-skill', name="decoration skill", description="Artistic finishing and traditional Chinese decoration", tags=['decoration', 'painting'])
decoration_card = AgentCard(name='decoration_agent', description='Traditional decoration specialist', skills=[decoration_skill], capabilities=AgentCapabilities(), default_input_modes=['text'], default_output_modes=['text'], url='localhost', version='0.1.0')
decoration_agent = WorkerAgent(chain=decoration_chain, card=decoration_card, state_change_callback=state_callback)

# QA Worker
qa_skill = AgentSkill(id='qa-skill', name="qa skill", description="Quality assurance and safety testing", tags=['qa', 'safety'])
qa_card = AgentCard(name='qa_agent', description='Quality assurance specialist', skills=[qa_skill], capabilities=AgentCapabilities(), default_input_modes=['text'], default_output_modes=['text'], url='localhost', version='0.1.0')
qa_agent = WorkerAgent(chain=qa_chain, card=qa_card, state_change_callback=state_callback)

workers = [design_agent, procurement_agent, skeleton_agent, decoration_agent, qa_agent]


## Define the Planner Agent

The planner agent takes the main lantern-making query and generates a structured plan with steps, worker assignments, and dependencies.


In [ ]:
planner_system_prompt = """You are a project coordinator for making a traditional Chinese wood lantern.
Your job is to break down the lantern-making task into clear, sequential steps and assign each step to the appropriate worker.

Available workers:
1. design_agent - Conceptual design (style, dimensions, structural plan, light source)
2. procurement_agent - Material sourcing (wood types, tools, adhesives, finishes, hardware)
3. skeleton_agent - Frame construction (cutting, joining, assembling the lantern skeleton)
4. decoration_agent - Artistic finishing (painting, carving, paper mounting, tassels, lighting)
5. qa_agent - Quality assurance (safety testing, structural integrity, durability)

For each step, specify:
- worker: which worker handles this step (one of: design_agent, procurement_agent, skeleton_agent, decoration_agent, qa_agent)
- message: a clear instruction for the worker about what to do
- dependencies: list of indices of previous steps this step depends on (empty list if no dependencies)

Rules:
- Always start with design (step 0) since it informs all subsequent steps
- Procurement depends on design (needs to know what materials to source)
- Skeleton depends on design and procurement (needs design specs and materials)
- Decoration depends on design, procurement, and skeleton (needs all prior work)
- QA depends on everything (final inspection of completed lantern)
- Each worker should receive the original query context plus any relevant dependencies

Output your plan as a JSON array of objects with keys: worker, message, dependencies.
Example format:
[
  {"worker": "design_agent", "message": "Design a traditional Chinese wood lantern...", "dependencies": []},
  {"worker": "procurement_agent", "message": "Source all materials for the lantern...", "dependencies": [0]},
  {"worker": "skeleton_agent", "message": "Build the lantern frame...", "dependencies": [0, 1]},
  {"worker": "decoration_agent", "message": "Decorate and finish the lantern...", "dependencies": [0, 1, 2]},
  {"worker": "qa_agent", "message": "Test and inspect the finished lantern...", "dependencies": [0, 1, 2, 3]}
]

IMPORTANT: Return ONLY the JSON array, nothing else. No explanations, no markdown formatting.
"""

In [ ]:
planner_chain = ChatCausalMultiTurnsChain(
    model=(shared_tokenizer_1, shared_model_1),
    system_prompt=planner_system_prompt,
    user_prompt_template="{query}",
    device="cuda",
    max_new_tokens=4096,
)
planner_chain.final_response_as_context("plan")

planner_skill = AgentSkill(id='planner-skill', name="planner skill", description="Task decomposition and planning", tags=['planner'])
planner_card = AgentCard(name='planner_agent', description='Project planning coordinator', skills=[planner_skill], capabilities=AgentCapabilities(), default_input_modes=['text'], default_output_modes=['text'], url='localhost', version='0.1.0')
planner = WorkerAgent(chain=planner_chain, card=planner_card, state_change_callback=state_callback)

## Implement Plan Parsing Function

This function parses the planner's JSON output into a sequence of tuples: (step_message, worker_agent, dependencies).


In [ ]:
def parse_plan(message: AgentMessage, workers: Sequence[BaseAgent]) -> Sequence[Tuple[AgentMessage, BaseAgent, List]]:
    """
    Parse the planner's output into a sequence of (step_message, worker_agent, dependencies) tuples.
    Handles both JSON and free-form text from the LLM.
    """
    # The planner chain uses final_response_as_context("plan"), so the response is in context["plan"]
    if message.context is None or "plan" not in message.context:
        # Fallback: try to get from responses if context is not set
        if message.responses:
            plan_text = message.responses[-1][1]
        else:
            raise ValueError("No plan found in message.context or message.responses")
    else:
        plan_text = str(message.context["plan"])

    # Map worker names to agent objects
    worker_map = {agent.card.name: agent for agent in workers}

    # Try 1: Extract JSON array from the response
    json_match = re.search(r'\[.*\]', plan_text, re.DOTALL)
    if json_match:
        try:
            plan = json.loads(json_match.group())
            steps = []
            for step in plan:
                worker_name = step["worker"]
                if worker_name not in worker_map:
                    raise ValueError(f"Unknown worker: {worker_name}. Available: {list(worker_map.keys())}")
                worker = worker_map[worker_name]
                dependencies = step.get("dependencies", [])
                step_msg = AgentMessage(query=step["message"])
                steps.append((step_msg, worker, dependencies))
            return steps
        except (json.JSONDecodeError, KeyError):
            pass  # Fall through to free-form parsing

    # Try 2: Parse free-form text by looking for worker assignments
    steps = []
    worker_names = list(worker_map.keys())
    lines = plan_text.split('\n')

    current_step = None
    step_index = 0

    for line in lines:
        line_lower = line.lower().strip()

        # Check if this line mentions a worker
        for worker_name in worker_names:
            if worker_name in line_lower:
                # Extract the message (everything after the worker name on this line or next few lines)
                msg_start = line_lower.find(worker_name) + len(worker_name)
                msg = line[msg_start:].strip().lstrip(':').strip()

                # If message is empty, look at next lines
                if not msg:
                    for next_line in lines[lines.index(line)+1:lines.index(line)+4]:
                        next_line = next_line.strip()
                        if next_line and not any(kw in next_line.lower() for kw in worker_names):
                            msg = next_line
                            break

                if msg:
                    # Try to extract dependencies from the line
                    deps_match = re.search(r'dependencies?\s*[:=]?\s*\[?(\d+(?:\s*,\s*\d+)*)\]?', line, re.IGNORECASE)
                    dependencies = []
                    if deps_match:
                        dependencies = [int(d.strip()) for d in deps_match.group(1).split(',') if d.strip().isdigit()]

                    step_msg = AgentMessage(query=msg)
                    steps.append((step_msg, worker_map[worker_name], dependencies))
                    step_index += 1
                    break

    if steps:
        return steps

    # Try 3: Fallback - create a single step for each worker with the full query
    steps = []
    for worker_name, worker in worker_map.items():
        step_msg = AgentMessage(query=f"Work on: {message.query}")
        steps.append((step_msg, worker, []))

    return steps


## Define Summary Agent for Final Synthesis

The summary agent aggregates results from all workers to produce a cohesive final lantern-making guide.


In [ ]:
summary_system_prompt = """You are a master woodworker and lantern-making artisan with 30 years of experience.
Your job is to synthesize all the design, procurement, construction, decoration, and QA reports into a comprehensive, cohesive lantern-making guide.

You will receive:
- The original lantern-making query
- Design plan (context_design)
- Materials list (context_procurement)
- Frame construction guide (context_skeleton)
- Decoration guide (context_decoration)
- Quality assurance checklist (context_qa)

Create a comprehensive final guide that:
1. Provides an overview of the lantern project and its cultural significance
2. Summarizes the design concept with key specifications
3. Lists all materials and tools needed in an organized shopping list
4. Provides step-by-step construction instructions in logical order
5. Includes detailed decoration and finishing instructions
6. Presents the quality assurance checklist
7. Offers tips for beginners and common mistakes to avoid
8. Suggests ways to personalize the lantern

Format the guide with clear headings, numbered steps, and actionable recommendations. Make it accessible to both beginners and experienced woodworkers.
"""

summary_chain = ChatCausalMultiTurnsChain(
    model=(shared_tokenizer_2, shared_model_2),
    system_prompt=summary_system_prompt,
    user_prompt_template="{query}\n\n{context_results}",
    device="cuda",
    max_new_tokens=4096,
)
summary_chain.final_response_as_context("summary")

summary_skill = AgentSkill(id='summary-skill', name="summary skill", description="Guide synthesis and report generation", tags=['summary'])
summary_card = AgentCard(name='summary_agent', description='Guide synthesis coordinator', skills=[summary_skill], capabilities=AgentCapabilities(), default_input_modes=['text'], default_output_modes=['text'], url='localhost', version='0.1.0')
summary_agent = WorkerAgent(chain=summary_chain, card=summary_card, state_change_callback=state_callback)


## Instantiate and Configure CoordinatorAgent

Combine the planner, workers, summary chain, and parsing function into a single CoordinatorAgent instance.


In [ ]:
# Coordinator card
coordinator_skill = AgentSkill(id='coordinator-skill', name="coordinator skill", description="Orchestrates multi-agent lantern making", tags=['coordinator'])
coordinator_card = AgentCard(name='coordinator_agent', description='Multi-agent lantern making coordinator', skills=[coordinator_skill], capabilities=AgentCapabilities(), default_input_modes=['text'], default_output_modes=['text'], url='localhost', version='0.1.0')

# Create the CoordinatorAgent
coordinator = CoordinatorAgent(
    planner_agent=planner,
    parse_plan=parse_plan,
    workers=workers,
    summary_chain=summary_chain,
    summary_steps_key="context_results",
    card=coordinator_card,
    state_change_callback=state_callback,
)


## Execute the Coordinator

Now let's run the coordinator with a query about making a traditional Chinese wood lantern.


In [ ]:
# Create the initial message
query = """Design and make a traditional Chinese wood lantern inspired by Fuzhou-style "umbrella lanterns" with the following requirements:
- Traditional red and gold color scheme with family name painted on one side
- Bamboo splint frame with umbrella-like structure that can fold for storage
- Rice paper covering with hand-painted dragon motif
- LED candle light source for safety (no open flame)
- Red string handle with gold tassels at the base
- Approximately 30cm height, suitable for indoor display
- Budget-friendly materials suitable for a hobbyist woodworker"""

message = AgentMessage(query=query)

# Execute the coordinator
result = coordinator.execute(message)


## View the Results

Display the final synthesized lantern-making guide from the coordinator.


In [ ]:
# Display the final result
print("=== Coordinator Execution Complete ===")
print(f"Execution result: {result.execution_result}")
print(f"Origin: {result.origin}")
print()

# Show the summary if available
if result.context and "summary" in result.context:
    print("=== FINAL LANTERN-MAKING GUIDE ===")
    print(result.context["summary"])
elif result.responses:
    print("=== FINAL RESPONSE ===")
    print(result.responses[-1][1])
else:
    print("No summary or responses available.")
    print(f"Context keys: {list(result.context.keys()) if result.context else 'None'}")
